# Task 2 Colab Training (Simple)

This notebook is Task 2 only. It trains the model and exports model.pth for Task 3 upload.

Use this on low-spec machines via Google Colab.

In [ ]:
# Runtime mode: change this to "local" on your own machine.
RUN_ENV = "colab"
IS_COLAB = RUN_ENV.lower() == "colab"

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/<your-org-or-user>/AAI.git"
if IS_COLAB:
    REPO_DIR = Path("/content/AAI")
    DATASET_DIR = Path("/content/task2_data")
    EXPORT_DIR = Path("/content/task2_export")
else:
    REPO_DIR = Path.cwd().resolve()
    if not (REPO_DIR / "task2_quality").exists() and (REPO_DIR.parent / "task2_quality").exists():
        REPO_DIR = REPO_DIR.parent.resolve()
    DATASET_DIR = Path(os.environ.get("TASK2_DATASET_DIR", str(REPO_DIR / "FruitAndVegetableDataset")))
    EXPORT_DIR = REPO_DIR / "task2_export"

print("RUN_ENV:", RUN_ENV)
print("REPO_DIR:", REPO_DIR)
print("DATASET_DIR:", DATASET_DIR)
print("EXPORT_DIR:", EXPORT_DIR)

In [ ]:
# Install base tools when running in Colab.
if IS_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
else:
    print("Local mode: skip kaggle install if already available in your environment.")

In [ ]:
# Configure repo location
if IS_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    if not (REPO_DIR / "task2_quality").exists():
        raise RuntimeError(f"Local mode expects to run from the AAI repo root. Current repo path: {REPO_DIR}")

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
# Install project requirements when needed.
if IS_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    print("Local mode: assume project requirements are already installed.")

In [ ]:
# Kaggle auth + optional dataset download.
import json
from getpass import getpass

DATASET_SLUG = "muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten"

if not DATASET_DIR.exists():
    if IS_COLAB or os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY"):
        kaggle_username = os.environ.get("KAGGLE_USERNAME") or input("Kaggle username: ").strip()
        kaggle_key = os.environ.get("KAGGLE_KEY") or getpass("Kaggle API key: ")

        os.environ["KAGGLE_USERNAME"] = kaggle_username
        os.environ["KAGGLE_KEY"] = kaggle_key

        os.makedirs("/root/.kaggle", exist_ok=True)
        with open("/root/.kaggle/kaggle.json", "w", encoding="utf-8") as f:
            json.dump({"username": kaggle_username, "key": kaggle_key}, f)
        os.chmod("/root/.kaggle/kaggle.json", 0o600)

        download_root = DATASET_DIR if IS_COLAB else DATASET_DIR.parent / "task2_data"
        download_root.mkdir(parents=True, exist_ok=True)
        subprocess.run(["kaggle", "datasets", "download", "-d", DATASET_SLUG, "-p", str(download_root), "--unzip"], check=True)
        print("Dataset downloaded to", download_root)
    else:
        raise RuntimeError("Dataset not found. Set TASK2_DATASET_DIR or place the dataset locally before training.")
else:
    print("Dataset already present:", DATASET_DIR)

# Find the actual image-folder root inside the download location.
search_root = DATASET_DIR
if search_root.is_file():
    search_root = search_root.parent

def score_dir(p: Path) -> int:
    children = [x for x in p.iterdir() if x.is_dir()]
    if len(children) < 2:
        return -1
    names = " ".join([c.name.lower() for c in children])
    bonus = 2 if ("healthy" in names or "rotten" in names) else 0
    return len(children) + bonus

candidates = [p for p in search_root.rglob("*") if p.is_dir()]
best = max(candidates, key=score_dir) if candidates else None
if best is None or score_dir(best) < 0:
    if search_root.is_dir() and score_dir(search_root) >= 0:
        best = search_root
    else:
        raise RuntimeError("Could not detect dataset directory. Check the download folder manually.")

DATASET_DIR = best
print("Detected DATASET_DIR:", DATASET_DIR)

In [ ]:
# Train using the production script.
EPOCHS = 12
BATCH_SIZE = 32
LR = 0.001

cmd = [
    sys.executable,
    "task2_quality/training_pipeline.py",
    "--dataset-dir", str(DATASET_DIR),
    "--model-version", "auto",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--learning-rate", str(LR),
    "--pretrained",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Resolve newest trained version and export files.
def parse_semver(name: str):
    parts = name.split(".")
    if len(parts) != 3:
        return None
    try:
        return tuple(int(x) for x in parts)
    except ValueError:
        return None

model_root = REPO_DIR / "models" / "produce-quality"
versions = []
for child in model_root.iterdir() if model_root.exists() else []:
    if child.is_dir():
        parsed = parse_semver(child.name)
        if parsed is not None:
            versions.append((parsed, child.name))

if not versions:
    raise RuntimeError("No semantic model versions found after training.")

trained_version = sorted(versions, key=lambda x: x[0])[-1][1]
artifact_path = model_root / trained_version / "artifacts" / "model.pth"
plot_path = REPO_DIR / "docs" / "task2" / f"task2_learning_curves_produce-quality_{trained_version}.png"
if not artifact_path.exists():
    raise RuntimeError(f"Expected artifact missing: {artifact_path}")

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
(EXPORT_DIR / "model.pth").write_bytes(artifact_path.read_bytes())
(EXPORT_DIR / "metadata.json").write_text(json.dumps({
    "trained_version": trained_version,
    "artifact_source": str(artifact_path),
    "plot_path": str(plot_path),
}, indent=2), encoding="utf-8")

print("trained_version:", trained_version)
print("artifact_path:", artifact_path)
print("export_dir:", EXPORT_DIR)

In [ ]:
# Download outputs only when running in Colab.
if IS_COLAB:
    from google.colab import files
    files.download(str(EXPORT_DIR / "model.pth"))
    files.download(str(EXPORT_DIR / "metadata.json"))
    if plot_path.exists():
        files.download(str(plot_path))
else:
    print("Local mode export paths:")
    print(EXPORT_DIR / "model.pth")
    print(EXPORT_DIR / "metadata.json")
    if plot_path.exists():
        print(plot_path)